# Unemployment Analysis with Python

### Exploratory Data Analysis, COVID-19 Impact, Trends, Seasonality & Policy Insights

This project analyzes unemployment in India using monthly regional observations.

> **Main question:** What changed in unemployment over time, where did the change happen, and what patterns can help inform employment policy?

The notebook combines data cleaning, descriptive exploration, interactive visualization, COVID-period comparison, regional analysis, monthly pattern exploration, relationship analysis, limitations, and policy-oriented insights.

### Project requirements covered
- Analyze unemployment-rate data.
- Clean and validate the dataset using Python.
- Explore and visualize unemployment trends over time.
- Investigate the unemployment changes associated with the COVID-19 period.
- Explore regional, rural/urban, and monthly patterns.
- Identify relationships between unemployment, employment, and labour participation.
- Present evidence-based findings that can inform economic or social policy.

> **Important note:** The COVID analysis is observational. This dataset can show how unemployment changed around the COVID period, but it cannot prove that COVID alone caused every observed change.

## 1. Project Roadmap

**Load → Inspect → Clean → Validate → Explore → Trend → Compare → Explain → Recommend**

Every visualization has a specific analytical purpose. Interactive charts are used where hover details, zooming, dropdown selection, or time animation adds real value.

## 2. Libraries

We use:
- **Pandas / NumPy** for data preparation and numerical analysis.
- **Matplotlib / Seaborn** for compact analytical plots.
- **Plotly** for interactive charts with hover, zoom, dropdowns, and animation.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

## 3. Load the Dataset

The loader first checks the expected task filename and then falls back to the uploaded dataset filename or a single unemployment CSV found in the working folder.

This keeps the notebook portable between VS Code, Jupyter, and Colab-style environments.

In [ ]:
df = pd.read_csv("Unemployment in India.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (768, 7)


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,"11,999,139.00",43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,"11,755,881.00",42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,"12,086,707.00",43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,"12,285,693.00",43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,"12,256,762.00",44.68,Rural


## 4. First Look at the Data

Before cleaning, we inspect the raw structure. This prevents us from making assumptions about missing values, duplicated observations, data types, and column names.

In [ ]:
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset shape:")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,"11,999,139.00",43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,"11,755,881.00",42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,"12,086,707.00",43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,"12,285,693.00",43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,"12,256,762.00",44.68,Rural



Data types:


,dtype
Region,str
Date,str
Frequency,str
Estimated Unemployment Rate (%),float64
Estimated Employed,float64
Estimated Labour Participation Rate (%),float64
Area,str



Dataset shape:
Rows    : 768
Columns : 7


In [ ]:
print("Raw descriptive summary:")
display(df.describe(include="all").T)

Raw descriptive summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,740,28,Andhra Pradesh,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,740,14,31-10-2019,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Frequency,740,2,Monthly,381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Estimated Unemployment Rate (%),740.00,NaN,NaN,NaN,11.79,10.72,0.00,4.66,8.35,15.89,76.74
Estimated Employed,740.00,NaN,NaN,NaN,"7,204,460.03","8,087,988.43","49,420.00","1,190,404.50","4,744,178.50","11,275,489.50","45,777,509.00"
Estimated Labour Participation Rate (%),740.00,NaN,NaN,NaN,42.63,8.11,13.33,38.06,41.16,45.51,72.57
Area,740,2,Urban,381,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

C:\Users\Khladoun\AppData\Local\Temp\ipykernel_14832\1126013313.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### What are we checking?

- **Rows and columns:** the size of the analytical dataset.
- **Data types:** whether dates and numerical measures are represented correctly.
- **Descriptive statistics:** the scale, center, and spread of numerical variables.
- **Preview rows:** whether values look structurally correct.

## 5. Data Quality Audit

The source file contains missing observations and repeated rows. Both issues are checked explicitly because they can influence averages and rankings.

In [ ]:
df.columns = [str(c).strip() for c in df.columns]

missing_before = df.isna().sum().sort_values(ascending=False)
duplicate_before = int(df.duplicated().sum())

print("Missing values before cleaning:")
display(missing_before.to_frame("Missing Values"))
print(f"Duplicate rows before cleaning: {duplicate_before}")

Missing values before cleaning:


,Missing Values
Region,28
Date,28
Frequency,28
Estimated Unemployment Rate (%),28
Estimated Employed,28
Estimated Labour Participation Rate (%),28
Area,28


Duplicate rows before cleaning: 27


In [ ]:
fig = px.imshow(
    df.isna().astype(int),
    aspect="auto",
    color_continuous_scale=["white", "#d95f59"],
    labels={"x": "Columns", "y": "Rows", "color": "Missing"},
    title="Missing-Value Map Before Cleaning")
fig.update_layout(height=450)
fig.show()

### Why this chart?

The heatmap shows **where** missing values occur rather than giving only a total count. This helps distinguish isolated missing cells from observations that are incomplete across several columns.

## 6. Cleaning and Standardization

Cleaning decisions:
1. Trim extra spaces.
2. Rename columns into concise, readable names.
3. Convert dates to `datetime`.
4. Convert numerical measures to numeric types.
5. Remove exact duplicate rows.
6. Remove rows missing a required analytical field.

No missing value is replaced with an invented estimate.

In [ ]:
rename_map = {    "Region": "State",
    "Date": "Date",
    "Frequency": "Frequency",
    "Estimated Unemployment Rate (%)": "Unemployment_Rate",
    "Estimated Employed": "Employed",
    "Estimated Labour Participation Rate (%)": "Labour_Participation_Rate",
    "Area": "Area"}


df = df.rename(columns=rename_map)

for col in ["State", "Frequency", "Area"]:
    df[col] = df[col].astype("string").str.strip()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

numeric_cols = ["Unemployment_Rate",
    "Employed",
    "Labour_Participation_Rate"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rows_before_cleaning = len(df)
df = df.drop_duplicates().copy()
required_cols = ["State", "Date", "Unemployment_Rate", "Employed",
    "Labour_Participation_Rate", "Area"]
df = df.dropna(subset=required_cols).copy()
rows_removed = rows_before_cleaning - len(df)
print(f"Rows removed during cleaning: {rows_removed}")
print(f"Rows remaining: {len(df)}")

Rows removed during cleaning: 28
Rows remaining: 740


In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")
df["Month_Label"] = df["Date"].dt.strftime("%b %Y")

df["Period"] = np.where(
    df["Date"] < pd.Timestamp("2020-03-01"),
    "Pre-COVID",
    "COVID period")

print("Final columns:")
print(df.columns.tolist())

Final columns:
['State', 'Date', 'Frequency', 'Unemployment_Rate', 'Employed', 'Labour_Participation_Rate', 'Area', 'Year', 'Month', 'Month_Name', 'Month_Label', 'Period']


## 7. Final Data Validation

A reliable analysis confirms that the cleaning step really solved the detected problems.

In [ ]:
print("Missing values after cleaning:")
display(df[required_cols].isna().sum().to_frame("Missing"))

print(f"Duplicate rows after cleaning: {df.duplicated().sum()}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

print("\nArea categories:")
display(df["Area"].value_counts().to_frame("Count"))

print("\nFrequency categories:")
display(df["Frequency"].value_counts().to_frame("Count"))

Missing values after cleaning:


,Missing
State,0
Date,0
Unemployment_Rate,0
Employed,0
Labour_Participation_Rate,0
Area,0


Duplicate rows after cleaning: 0
Date range: 2019-05-31 → 2020-06-30

Area categories:


,Count
Area,
Urban,381
Rural,359



Frequency categories:


,Count
Frequency,
Monthly,740


## 8. Dataset Overview

This section creates a compact baseline for the main indicators before detailed analysis.

In [ ]:
overview = pd.DataFrame({"Metric": [
        "Rows",
        "States/Regions",
        "Months represented",
        "Average unemployment rate (%)",
        "Median unemployment rate (%)",
        "Average labour participation rate (%)",
        "Average employed"],

    "Value": [
        len(df),
        df["State"].nunique(),
        df["Date"].nunique(),
        df["Unemployment_Rate"].mean(),
        df["Unemployment_Rate"].median(),
        df["Labour_Participation_Rate"].mean(),
        df["Employed"].mean()] })

display(overview)

,Metric,Value
0,Rows,740.00
1,States/Regions,28.00
2,Months represented,14.00
3,Average unemployment rate (%),11.79
4,Median unemployment rate (%),8.35
5,Average labour participation rate (%),42.63
6,Average employed,"7,204,460.03"


## 9. Overall Unemployment Trend Over Time

### Purpose
This is the core trend analysis required by the task. We want to see the direction of unemployment, the highest and lowest monthly averages, and the shift around the COVID period.

### Interaction
Hover for exact values, zoom using the chart, and use the range slider to focus on a shorter period.

In [ ]:
monthly_trend = (
    df.groupby("Date", as_index=False)["Unemployment_Rate"]
      .mean()
      .sort_values("Date"))

fig = px.line(
    monthly_trend,
    x="Date",
    y="Unemployment_Rate",
    markers=True,
    title="Overall Monthly Unemployment Rate",
    labels={"Date": "Month", "Unemployment_Rate": "Average Unemployment Rate (%)"},
    template="plotly_white")

fig.add_vrect(
    x0=pd.Timestamp("2020-03-01"),
    x1=df["Date"].max(),
    fillcolor="red",
    opacity=0.08,
    line_width=0,
    annotation_text="COVID period",
    annotation_position="top left" )

fig.update_traces(hovertemplate="%{x|%b %Y}<br>Unemployment: %{y:.2f}%<extra></extra>")
fig.update_layout(
    height=520,
    hovermode="x unified",
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=6, label="6M", step="month", stepmode="backward"),
            dict(count=12, label="12M", step="month", stepmode="backward"),
            dict(step="all", label="All") ]) ) )
fig.show()

In [ ]:
peak_row = monthly_trend.loc[monthly_trend["Unemployment_Rate"].idxmax()]
lowest_row = monthly_trend.loc[monthly_trend["Unemployment_Rate"].idxmin()]

display(Markdown(
    f"""
**Trend takeaway:** The highest monthly average unemployment rate in the cleaned dataset was
**{peak_row['Unemployment_Rate']:.2f}%** in **{peak_row['Date']:%b %Y}**, while the lowest was
**{lowest_row['Unemployment_Rate']:.2f}%** in **{lowest_row['Date']:%b %Y}**.
"""
))


**Trend takeaway:** The highest monthly average unemployment rate in the cleaned dataset was
**24.88%** in **May 2020**, while the lowest was
**8.87%** in **May 2019**.


## 10. Interactive State Trend Explorer

### Purpose
The national average can hide regional differences. This dropdown chart allows the reader to select one state and inspect its unemployment path through time.

Instead of creating many separate static charts, one interactive figure supports targeted exploration.

In [ ]:
states_sorted = (
    df.groupby("State")["Unemployment_Rate"]
      .mean()
      .sort_values(ascending=False)
      .index.tolist())

fig = go.Figure()
for state in states_sorted:
    temp = (
        df[df["State"] == state]
        .groupby("Date", as_index=False)["Unemployment_Rate"]
        .mean()
        .sort_values("Date"))
    fig.add_trace(
        go.Scatter(
            x=temp["Date"],
            y=temp["Unemployment_Rate"],
            mode="lines+markers",
            name=state,
            visible=False,
            hovertemplate=f"{state}<br>%{{x|%b %Y}}<br>Unemployment: %{{y:.2f}}%<extra></extra>" ) )

fig.data[0].visible = True

buttons = []
for i, state in enumerate(states_sorted):
    visibility = [False] * len(states_sorted)
    visibility[i] = True
    buttons.append(
        dict(
            label=state,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Unemployment Trend — {state}"} ]  ))

fig.update_layout(
    title=f"Unemployment Trend — {states_sorted[0]}",
    template="plotly_white",
    height=550,
    xaxis=dict(title="Month", rangeslider=dict(visible=True)),
    yaxis_title="Unemployment Rate (%)",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=1.02,
        xanchor="left",
        y=1,
        yanchor="top" )] )
fig.show()

## 11. State-Level Unemployment Ranking

### Purpose
This chart answers: **Which states have the highest average unemployment rates across the observation period?**

It is useful for identifying regions that may require closer labour-market investigation.

In [ ]:
state_avg = (
    df.groupby("State")["Unemployment_Rate"]
      .mean()
      .sort_values(ascending=False)
      .reset_index(name="Average_Unemployment"))

fig = px.bar(
    state_avg.head(15).sort_values("Average_Unemployment"),
    x="Average_Unemployment",
    y="State",
    orientation="h",
    text="Average_Unemployment",
    title="Top 15 States by Average Unemployment Rate",
    labels={"Average_Unemployment": "Average Unemployment Rate (%)", "State": "State / Region"},
    template="plotly_white")

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
    hovertemplate="%{y}<br>Average unemployment: %{x:.2f}%<extra></extra>")
fig.update_layout(height=620)
fig.show()

### Interpretation

This is a screening ranking, not a judgement about a state. A high average can reflect different local labour-market conditions and the composition of observations in the dataset.

## 12. Rural vs Urban Unemployment

### Purpose
The `Area` variable separates rural and urban observations. We compare both the average level and the distribution because a mean alone does not show variability.

In [ ]:
area_summary = (
    df.groupby("Area")
      .agg(
          Average_Unemployment=("Unemployment_Rate", "mean"),
          Median_Unemployment=("Unemployment_Rate", "median"),
          Observations=("Unemployment_Rate", "size"))
      .sort_values("Average_Unemployment", ascending=False) )

display(area_summary.round(2))

fig = px.box(
    df,
    x="Area",
    y="Unemployment_Rate",
    points="outliers",
    color="Area",
    title="Unemployment Rate Distribution by Area Type",
    labels={"Area": "Area Type", "Unemployment_Rate": "Unemployment Rate (%)"},
    template="plotly_white")
fig.update_layout(showlegend=False, height=500)
fig.show()

,Average_Unemployment,Median_Unemployment,Observations
Area,,,
Urban,13.17,9.97,381
Rural,10.32,6.76,359


### Why a box plot?

The box plot shows the **median, spread, and unusual observations** for each area type. This makes the comparison more informative than a single bar.

## 13. COVID-19 Period Comparison

### Definition
- **Pre-COVID:** before 1 March 2020
- **COVID period:** 1 March 2020 onward

This is a transparent before/after split for the available monthly observations. It is not a causal experiment.

In [ ]:
period_summary = (
    df.groupby("Period")["Unemployment_Rate"]
      .agg(["mean", "median", "min", "max", "count"])
      .reindex(["Pre-COVID", "COVID period"]))

display(period_summary.round(2))

,mean,median,min,max,count
Period,,,,,
Pre-COVID,9.51,7.12,0.00,34.69,536
COVID period,17.77,14.52,0.00,76.74,204


In [ ]:
fig = px.box(
    df,
    x="Period",
    y="Unemployment_Rate",
    color="Period",
    points="outliers",
    title="Unemployment Rate Before and During the COVID Period",
    labels={"Period": "Period", "Unemployment_Rate": "Unemployment Rate (%)"},
    template="plotly_white")
fig.update_layout(showlegend=False, height=520)
fig.show()

In [ ]:
pre_mean = period_summary.loc["Pre-COVID", "mean"]
covid_mean = period_summary.loc["COVID period", "mean"]
absolute_change = covid_mean - pre_mean
percent_change = (absolute_change / pre_mean) * 100

display(Markdown(
    f"""
**COVID-period comparison:** The average unemployment rate changed from **{pre_mean:.2f}%**
before COVID to **{covid_mean:.2f}%** during the COVID period.

That is an absolute change of **{absolute_change:+.2f} percentage points**
({percent_change:+.1f}% relative to the pre-COVID average).

This is evidence of a large change around the COVID period in this dataset, not proof of a single causal mechanism.
"""
))


**COVID-period comparison:** The average unemployment rate changed from **9.51%**
before COVID to **17.77%** during the COVID period.

That is an absolute change of **+8.26 percentage points**
(+86.9% relative to the pre-COVID average).

This is evidence of a large change around the COVID period in this dataset, not proof of a single causal mechanism.


## 14. Where Did Unemployment Change the Most?

### Purpose
A national change does not affect every region equally. Here we compare each state's pre-COVID and COVID-period averages.

A positive value means unemployment was higher during the COVID period.

In [ ]:
state_period = (
    df.groupby(["State", "Period"])["Unemployment_Rate"]
      .mean()
      .unstack())

state_period["Change_pp"] = state_period["COVID period"] - state_period["Pre-COVID"]

largest_increases = state_period.sort_values("Change_pp", ascending=False).head(10)
largest_decreases = state_period.sort_values("Change_pp", ascending=True).head(10)

display(
    pd.concat([
        largest_increases[["Pre-COVID", "COVID period", "Change_pp"]].rename_axis("State").reset_index().assign(Group="Largest increases"),
        largest_decreases[["Pre-COVID", "COVID period", "Change_pp"]].rename_axis("State").reset_index().assign(Group="Largest decreases")
    ]).round(2))

Period,State,Pre-COVID,COVID period,Change_pp,Group
0,Puducherry,1.59,38.96,37.36,Largest increases
1,Tamil Nadu,2.84,25.40,22.57,Largest increases
2,Jharkhand,14.28,36.35,22.07,Largest increases
3,Bihar,13.83,31.63,17.80,Largest increases
4,Karnataka,3.23,15.28,12.05,Largest increases
5,Haryana,22.94,34.65,11.72,Largest increases
6,Kerala,6.99,17.95,10.96,Largest increases
7,Telangana,4.66,15.44,10.79,Largest increases
8,Madhya Pradesh,4.74,14.07,9.33,Largest increases
9,Andhra Pradesh,5.04,13.58,8.54,Largest increases


In [ ]:
change_plot = (
    pd.concat([
        largest_increases.reset_index().assign(Group="Largest increases"),
        largest_decreases.reset_index().assign(Group="Largest decreases")
    ])
    .sort_values("Change_pp"))

fig = px.bar(
    change_plot,
    x="Change_pp",
    y="State",
    color="Group",
    orientation="h",
    text="Change_pp",
    title="States with the Largest COVID-Period Changes",
    labels={
        "Change_pp": "Change in unemployment (percentage points)",
        "State": "State / Region",
        "Group": "Ranking group"},
    template="plotly_white")
fig.update_traces(
    texttemplate="%{text:+.2f}",
    textposition="outside",
    hovertemplate="%{y}<br>Change: %{x:+.2f} percentage points<extra></extra>")
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=620)
fig.show()

### Why this analysis is useful

The overall COVID comparison tells us **that** unemployment changed.  
The state-level change analysis tells us **where the change was concentrated**. That makes the policy discussion more targeted.

## 15. Monthly / Seasonal Pattern Exploration

### Purpose
The task asks for seasonal or recurring patterns. We therefore calculate average unemployment for each calendar month.

### Limitation
The dataset covers only about 14 months, so there are not enough repeated years to prove stable seasonality. This section is exploratory rather than a definitive seasonal model.

In [ ]:
monthly_seasonal = (
    df.groupby("Month")["Unemployment_Rate"]
      .mean()
      .reindex(range(1, 13))
      .dropna())

month_names = [
    pd.Timestamp(year=2020, month=i, day=1).strftime("%b")
    for i in monthly_seasonal.index]

seasonal_plot = pd.DataFrame({
    "Month": month_names,
    "Average_Unemployment": monthly_seasonal.values})

fig = px.bar(
    seasonal_plot,
    x="Month",
    y="Average_Unemployment",
    text="Average_Unemployment",
    title="Average Unemployment Rate by Calendar Month",
    labels={
        "Month": "Calendar month",
        "Average_Unemployment": "Average Unemployment Rate (%)"
    },
    template="plotly_white")

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside")

fig.update_layout(height=500)
fig.show()

## 16. Interactive Animated Labour-Market Relationship

### Purpose

This is the notebook's main dashboard-style visualization.

- **X-axis:** Labour participation rate
- **Y-axis:** Unemployment rate
- **Bubble size:** estimated employed population
- **Color:** Rural / Urban
- **Animation:** month
- **Hover:** state-level details

The Play button and timeline slider make it possible to observe how the labour-market picture evolves month by month.

In [ ]:
bubble_df = df.copy()

bubble_df["Date_Label"] = bubble_df["Date"].dt.strftime("%b %Y")

fig = px.scatter(
    bubble_df,

    x="Labour_Participation_Rate",
    y="Unemployment_Rate",
    size="Employed",
    color="Area",
    hover_name="State",
    hover_data={
        "Labour_Participation_Rate": ":.2f",
        "Unemployment_Rate": ":.2f",
        "Employed": ":,.0f",
        "Area": True,
        "Date_Label": True},

    animation_frame="Date_Label",
    size_max=45,
    title="Interactive Labour-Market Explorer",
    labels={
        "Labour_Participation_Rate": "Labour Participation Rate (%)",
        "Unemployment_Rate": "Unemployment Rate (%)",
        "Employed": "Estimated Employed"},
    template="plotly_white")


fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1500
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 800

fig.update_layout(
    height=650,

    xaxis=dict(
        range=[
            max(0, bubble_df["Labour_Participation_Rate"].min() - 2),
            bubble_df["Labour_Participation_Rate"].max() + 2 ]),

    yaxis=dict(
        range=[
            max(0, bubble_df["Unemployment_Rate"].min() - 2),
            bubble_df["Unemployment_Rate"].max() + 5 ]  ))

fig.show()

### Why this visualization earns its place

A static scatter plot shows the relationship for many observations at once, while the animation adds the missing **time dimension**. The result is an exploratory view of how unemployment and labour participation moved together across regions as the months changed.

## 17. Relationship Between Unemployment, Employment and Labour Participation

### Purpose
Unemployment is only one component of the labour market. Correlation analysis provides an initial view of how the three numerical indicators move together.

Correlation is descriptive here; it does not establish causation.

In [ ]:
corr_cols = [
    "Unemployment_Rate",
    "Employed",
    "Labour_Participation_Rate"]

corr = df[corr_cols].corr()
display(corr.round(3))

,Unemployment_Rate,Employed,Labour_Participation_Rate
Unemployment_Rate,1.00,-0.22,0.00
Employed,-0.22,1.00,0.01
Labour_Participation_Rate,0.00,0.01,1.00


In [ ]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix of Labour-Market Indicators",
    labels={"color": "Correlation"})
fig.update_layout(height=500)
fig.show()

### Reading the heatmap

- Values near **+1** indicate strong positive linear association.
- Values near **-1** indicate strong negative linear association.
- Values near **0** indicate weak linear association.

The matrix helps explain the structure of the dataset, but it is not a causal model.

## 18. Data-Driven Key Findings

The statements below are calculated from the cleaned dataset so the narrative stays consistent with the actual analysis.

In [ ]:
highest_state = state_avg.iloc[0]
highest_area_name = area_summary.index[0]
highest_month = seasonal_plot.iloc[seasonal_plot["Average_Unemployment"].idxmax()]
lowest_month = seasonal_plot.iloc[seasonal_plot["Average_Unemployment"].idxmin()]

display(Markdown(
    f"""
### Key Findings

1. **Overall trend:** The highest observed monthly average unemployment rate was
   **{peak_row['Unemployment_Rate']:.2f}%** in **{peak_row['Date']:%b %Y}**.

2. **Regional difference:** **{highest_state['State']}** had the highest average unemployment rate
   in the cleaned dataset at **{highest_state['Average_Unemployment']:.2f}%**.

3. **Area comparison:** The **{highest_area_name}** observations had the higher average unemployment
   rate between the two area types.

4. **COVID-period shift:** The difference between the COVID-period and pre-COVID averages was
   **{absolute_change:+.2f} percentage points**.

5. **Monthly pattern:** **{highest_month['Month']}** had the highest average unemployment among the
   calendar months represented, while **{lowest_month['Month']}** had the lowest.

6. **Overall interpretation:** Unemployment varies by time, region, and area type, supporting a
   targeted rather than one-size-fits-all policy approach.
"""
))


### Key Findings

1. **Overall trend:** The highest observed monthly average unemployment rate was
   **24.88%** in **May 2020**.

2. **Regional difference:** **Tripura** had the highest average unemployment rate
   in the cleaned dataset at **28.35%**.

3. **Area comparison:** The **Urban** observations had the higher average unemployment
   rate between the two area types.

4. **COVID-period shift:** The difference between the COVID-period and pre-COVID averages was
   **+8.26 percentage points**.

5. **Monthly pattern:** **Apr** had the highest average unemployment among the
   calendar months represented, while **Jul** had the lowest.

6. **Overall interpretation:** Unemployment varies by time, region, and area type, supporting a
   targeted rather than one-size-fits-all policy approach.


## 19. Economic and Social Policy Insights

These recommendations are tied to the dimensions measured in the notebook.

### 1. Target high-pressure regions
Regions with high average unemployment or large COVID-period increases can be prioritized for employment-support programs and closer labour-market monitoring.

### 2. Use area-specific interventions
Differences between rural and urban observations suggest that employment policy may need to be adapted to local conditions.

### 3. Strengthen crisis-response mechanisms
A sharp change around a major disruption supports the value of rapid employment support, reskilling, and job-matching systems during future shocks.

### 4. Monitor unemployment with labour participation
Tracking participation alongside unemployment provides a broader labour-market signal than unemployment alone.

### 5. Build regional early-warning dashboards
A monthly monitoring system can flag unusually large regional increases so that investigations and support can happen earlier.

> These are analytical recommendations based on this dataset, not causal policy evaluations.

## 20. Limitations

A complete data project should state what the data cannot establish.

- The observation period is short, so seasonal conclusions are exploratory.
- The COVID split is a before/after comparison and does not isolate causality.
- Correlation does not prove cause and effect.
- Regional averages can hide differences within states and within rural/urban groups.
- The notebook describes historical observations; it does not forecast future unemployment.

## 21. Conclusion

The project moved from raw-data validation to a multi-angle exploration of unemployment in India.

It:
- cleaned and validated the source data,
- measured the overall unemployment trend,
- compared states and area types,
- quantified the change around the COVID period,
- explored monthly patterns,
- examined relationships among labour-market indicators,
- and translated the findings into policy-oriented insights.

The central analytical message is that **unemployment is time-varying and regionally uneven**, so effective monitoring and intervention should consider both dimensions.